# 27_03 랜덤 포레스트 고장 임박 분류


In [4]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

Duplicate key in file WindowsPath('c:/Users/mzlap/Desktop/KNA-Data-analysis-1st/.venv/Lib/site-packages/matplotlib/mpl-data/matplotlibrc'), line 851 ('font.family : Malgun Gothic')
Duplicate key in file WindowsPath('c:/Users/mzlap/Desktop/KNA-Data-analysis-1st/.venv/Lib/site-packages/matplotlib/mpl-data/matplotlibrc'), line 852 ('axes.unicode_minus : False')


✅ 환경 설정 완료! 현재 적용된 폰트: ['Malgun Gothic']


## 모델 만들고 학습시키기
모듈 2에서 만든 X_train, y_train으로 랜덤 포레스트 학습

### 분류기 불러오기
`sklearn.ensemble`에서 `RandomForestClassifier` 임포트

In [23]:
# ==============================================================================
# 1. 데이터 로드 및 타겟 변수(y) 생성
# ==============================================================================
# C-MAPSS FD001 데이터셋 불러오기
df = pd.read_csv("27_cmapss_fd001_sample.csv")

# 타겟 변수 생성: 잔여수명(RUL)이 30 이하이면 고장 임박(1), 아니면 정상(0)
df["failure_soon"] = (df["RUL"] <= 30).astype(int)

# 타겟 클래스 불균형 확인 (고장 임박 vs 정상 데이터 개수 및 비율)
print("--- [클래스 개수 확인] ---")
print(df["failure_soon"].value_counts()) 

print("\n--- [클래스 비율 확인] ---")
print(df["failure_soon"].value_counts(normalize=True)) 


# ==============================================================================
# 2. 특성(X)과 타겟(y) 정의
# ==============================================================================
# 예측 모델 학습에 사용할 센서 피처 선택
feats = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11", "sensor_15"]

X = df[feats]            # 입력 특성 (Features)
y = df["failure_soon"]   # 예측할 정답 (Target)


# ==============================================================================
# 3. 데이터셋 분할 (Train / Test Split)
# ==============================================================================
# [분할 목적]
# - 학습용 데이터와 평가용 데이터를 엄격히 Seprate하여 오버피팅(암기)을 방지함
# - 모델이 학습 데이터만 외우지 않고, 새로운 데이터에 대해 일반화(응용)할 수 있는지 검증
#
# [파라미터 설명]
# - test_size = 0.2  : 전체 데이터 중 20%를 테스트용으로 분할
# - random_state = 42: 난수 고정으로 코드 실행 시마다 동일한 분할 결과 유지
# - stratify = y     : y(타겟)의 0과 1 비율을 Train과 Test에 동일하게 유지 (불균형 방지)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)


# ==============================================================================
# 4. 분할 결과 검증 및 출력
# ==============================================================================
print("\n==================== [데이터 차원(Shape) 확인] ====================")
print("X_train shape : ", X_train.shape)  # 학습용 피처 (행, 열)
print("X_test  shape : ", X_test.shape)   # 평가용 피처 (행, 열)
print("y_train shape : ", y_train.shape)  # 학습용 정답 (행,)
print("y_test  shape : ", y_test.shape)   # 평가용 정답 (행,)

print("\n==================== [샘플 데이터 확인 (상위 3개)] ====================")
print("==== X_train ====\n", X_train.head(3), "\n")
print("==== X_test  ====\n", X_test.head(3), "\n")
print("==== y_train ====\n", y_train.head(3), "\n")
print("==== y_test  ====\n", y_test.head(3))



--- [클래스 개수 확인] ---
failure_soon
0    2994
1     620
Name: count, dtype: int64

--- [클래스 비율 확인] ---
failure_soon
0    0.828445
1    0.171555
Name: proportion, dtype: float64

==================== [데이터 차원(Shape) 확인] ====================
X_train shape :  (2891, 6)
X_test  shape :  (723, 6)
y_train shape :  (2891,)
y_test  shape :  (723,)

==================== [샘플 데이터 확인 (상위 3개)] ====================
==== X_train ====
       sensor_2  sensor_3  sensor_4  sensor_7  sensor_11  sensor_15
1119   643.822  1585.899  1420.613   552.669     48.335      8.548
2128   643.090  1594.474  1412.536   552.391     48.028      8.576
2675   641.849  1582.238  1390.616   554.407     47.274      8.407 

==== X_test  ====
       sensor_2  sensor_3  sensor_4  sensor_7  sensor_11  sensor_15
1431   642.652  1586.659  1412.582   552.687     47.660      8.492
153    641.165  1583.786  1394.940   554.045     47.063      8.395
514    642.870  1588.340  1417.942   552.783     48.206      8.513 

==== y_train ====
 11

### 모델 생성
트리 100그루 모델 생성 — 아직 학습 전 빈 상자


In [ ]:
# 코드

### 학습 실행
학습용 데이터로 학습 — 평가용은 절대 넣지 않음


In [ ]:
# 코드

### 학습 확인
모델 변수를 출력해 학습이 끝났는지 확인


## 예측하고 결과 형태 확인
- 평가용 데이터로 예측 받고 결과가 0/1 배열임을 확인



### 예측 실행
예측 결과를 y_pred에 담고 출력 — 0/1로만 이루어진 배열인가?


In [ ]:
# 코드

### 개수 확인
예측 개수와 평가용 행 개수가 같은지 확인


In [ ]:
# 코드

### 앞부분 보기
예측의 앞 10개만 잘라 보기 — 순서는 X_test와 동일


In [ ]:
# 코드

## 결과 형태 정리
- 예측 = 행마다 0/1이 담긴 배열, 입력과 같은 순서
## predict_proba로 확신도 살펴보기
- 각 행의 0일 확률과 1일 확률 확인 — 강한 확신/애매한 행 비교


### 확률 받기
예측 확률을 받아 앞 5개 확인 — 두 확률을 더하면 1이 되는가?


In [ ]:
# 코드

### 1일 확률만 보기
두 번째 열, 즉 1일 확률(곧 고장 확률)만 따로 보기


In [ ]:
# 코드

## 확신도 해석
- 확률이 높은 행과 0.5 부근인 행을 비교해 확신도 차이를 느껴 보기
## random_state 바꿔 다시 학습
- 값을 바꾸면 결과가 조금 달라짐, 같은 값이면 똑같이 재현됨


### 다른 random_state로 학습
random_state를 7로 바꿔 새 모델을 학습하고 예측


In [ ]:
# 코드

### 예측 비교
앞 예측과 새 예측의 앞부분 비교 — 일부 예측이 조금 달라졌는가?

In [ ]:
# 코드

### 같은 값이면 재현
같은 random_state 42로 다시 학습해 결과가 똑같은지 확인


In [ ]:
# 코드

## 예측 비교 + feature importance
- 예측·실제 눈 비교 + 어떤 센서가 판단에 크게 쓰였는지 확인


### 예측·실제 나란히 보기
예측과 실제를 한 표에 담아 앞부분 확인


In [ ]:
# 코드

## 눈으로 일치 세기
- 표를 보며 예측과 실제가 같은 행이 많은지 확인


### 중요 센서 확인
각 센서의 중요도를 큰 순으로 정렬


In [ ]:
# 코드

## 중요도 활용
- 중요한 센서 부위를 우선 점검 안내 — 모델 판단이 현장 행동으로
## 예측 결과를 표로 모아 비교
- 예측·실제·곧고장확률을 한 표에 모아 정비 우선순위로 가공


### 결과 표 만들기
예측·실제·곧고장확률을 한 표에 — 확률은 1일 확률 사용


In [ ]:
# 코드

### 곧 고장만 추리기
1로 예측된 행만 추리기 — 우선 점검 후보


In [ ]:
# 코드

### 우선순위 정렬
곧 고장 확률 높은 순으로 정렬해 우선순위 만들기


In [ ]:
# 코드

## 표 활용 정리
- 정렬된 표 자체가 정비 우선순위 목록 — 그대로 정비팀 전달 가능
## MIMII feature CSV로 정상/이상 분류
- 소리 특징 데이터에 똑같은 분류 흐름 적용 — 데이터만 바뀔 뿐


### 소리 데이터 불러오기
`rms`, `spectral_centroid`, `zero_crossing_rate`, `label` 컬럼 확인

In [ ]:
# 코드

### X/y 나누기
소리 특징 세 개를 입력 mX로, label을 정답 my로 분리


In [ ]:
# 코드

### 학습용/평가용 분리
엔진 때와 똑같이 분리 — stratify로 정상/이상 비율 유지


In [ ]:
# 코드

### 학습과 예측
랜덤 포레스트로 학습하고 예측 — 코드 흐름이 엔진 때와 거의 동일


In [ ]:
# 코드

## 소리 분류 정리
- 데이터만 바뀌었을 뿐 흐름은 동일 — 분류의 본질은 같음
## 종합 미니 실습 (전체 파이프라인)
- 불러오기 → 라벨 → X/y → 분리 → 학습 → 예측 전 과정을 한 흐름으로


### 불러오기와 라벨
데이터를 불러오고 RUL로 failure_soon 라벨 만들기

In [ ]:
# 코드

### X/y 구성
센서 여섯 개를 X로, failure_soon을 y로 — RUL이 X에 없는지 확인


In [ ]:
# 코드

### 데이터 분리
학습용과 평가용으로 나누기 — stratify=y로 비율 유지


In [ ]:
# 코드

### 학습과 예측
랜덤 포레스트로 학습하고 예측 — 여섯 단계를 막힘 없이 이었나?


In [ ]:
# 코드

## 종합 완성 점검
- 여섯 단계를 스스로 완성 — 오류가 나면 어느 단계인지 찾아 고치기
## 종합 결과 읽고 정리
- 예측·실제·확신도·중요 센서를 정비 우선순위로 해석하며 마무리


### 결과 표와 눈비교
예측·실제·곧고장확률을 모아 앞부분 눈비교 — 점수는 다음 파트


In [ ]:
# 코드

### 우선순위와 중요 센서
곧 고장 확률 순 우선 점검 목록 + 어떤 센서가 크게 쓰였는지 확인


In [ ]:
# 코드

## 한 문장으로 정리
곧 고장 확률 높은 설비부터 점검, 중요 센서 부위 우선 살피기, 사람이 검토